# 06 Governance: tags, grants, a row filter and lineage

This notebook is the evidence for the **Unity Catalog** box. It applies the SQL files next to it (`tags.sql`, `row_filters.sql`, `grants.sql`), then reads the result back from Unity Catalog so that every claim below is something this run measured. It makes **no AI calls**; a plan check near the end asserts 0 `AI Query` steps.

## What is governed, and what is deliberately not

| Control | Where | Notes |
|---|---|---|
| Tags | 7 `bluerridge_*` schemas, 14 key managed tables, 3 columns | see "Tag-policy finding" |
| **Row filter** | `gold_pm.drawing_register` only | by `discipline`, driven by the mapping table `agents.discipline_access` |
| Column masks | none | |
| Grants | `grants.sql` is the source of truth | see "Groups outcome". `account users` already has broad inherited read access from the catalog (section 5b), so the row filter is the only access control this build adds |
| Lineage | read from `system.access.table_lineage` | committed as `outputs/lineage_edges.md` |

**No row filter or column mask is placed on `kpi_daily`, `compliance_findings` or `risk_forecast`** (Task 16 syncs them to Lakebase, and synced tables cannot have a filtered or masked source), **nor on any table a pipeline or notebook reads** (a filter there would silently filter the doc-intelligence and gold notebooks). The new table `drawing_register` is read by nothing in the build, only by the Genie space and the app, so it is the safe place to demonstrate row-level security. A check below asserts that exactly one row filter and zero column masks exist across the `bluerridge_*` schemas.

## Tag-policy finding

Probing the account before writing `tags.sql` showed that it enforces **governed tags** for the keys `layer`, `sensitivity`, `data_class` and `domain`, each with its own allowed values. Examples of the rejections (`UC_TAG_POLICY_VALUE_NOT_ALLOWED`):

- `sensitivity = low` rejected, allowed `pii | internal | public`
- `layer = app` rejected, allowed `bronze | silver | gold`
- `data_class = reference` rejected, allowed HR values such as `compensation | payroll | benefits`
- `domain = app` rejected, allowed `finance | sales | supply_chain | quality | hr | operations`

Those policies belong to the shared account, so the project does not change them. The project vocabulary is carried in **prefixed keys that no policy governs**: `br_layer`, `br_sensitivity`, `br_data_class`, `br_domain`, plus `synthetic = true`. Where a governed key accepts our value (`layer` = bronze, silver or gold on those schemas) it is set as well.

## Groups outcome

Before this run, `databricks groups create` succeeded for `bluerridge-pm` and `bluerridge-engineers` (both came back as `WorkspaceGroup`). A Unity Catalog grant of USE SCHEMA on `mfg_mc_se_sa.bluerridge_gold_pm` to the group `bluerridge-pm` was then rejected with `PRINCIPAL_DOES_NOT_EXIST: Could not find principal with name bluerridge-pm`: Unity Catalog resolves account-level groups, and creating those needs account admin, which this demo does not have. **Both probe groups were deleted** and this task made no direct grant to `account users` or any other broad principal. `grants.sql` holds the intended grants with principal placeholders and this notebook applies a statement only when a widget supplies the principal. With no principal supplied, 0 grants are applied here: the owner already holds everything.

## Row-level security design

`discipline_access(principal, discipline)` maps a user to a discipline, where `'*'` means all. `discipline_allowed(principal, discipline)` returns true for a `'*'` row or an exact match. `rf_drawing_discipline(discipline)` is the row filter and just calls `discipline_allowed(current_user(), discipline)`. The mapping uses two fictitious principals (`mech.engineer@example.com`, `piping.lead@example.com`), which cannot log in, so what they would see is shown by calling the helper on the unfiltered silver table. A drawing whose title block gave no discipline (2 of 1,200) is hidden from everyone except a `'*'` principal.

**TODO for Task 24:** after the app deploys, add its service principal to `discipline_access` with `'*'` (statement in `grants.sql`), otherwise the app sees 0 rows of `drawing_register`.

In [1]:
import contextlib, io, re
from pathlib import Path
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "mfg_mc_se_sa", "Catalog")
dbutils.widgets.text("schema_prefix", "bluerridge", "Schema prefix")
dbutils.widgets.text("pm_group", "", "Account group for project managers (empty = not applied)")
dbutils.widgets.text("engineer_group", "", "Account group for engineers (empty = not applied)")
dbutils.widgets.text("app_sp", "", "App service principal application id (empty = not applied, Task 24)")
dbutils.widgets.text("run_id", "", "Job run id (set by the job as {{job.run_id}})")
catalog = dbutils.widgets.get("catalog"); prefix = dbutils.widgets.get("schema_prefix")
PARAMS = {"pm_group": dbutils.widgets.get("pm_group"), "engineer_group": dbutils.widgets.get("engineer_group"), "app_sp": dbutils.widgets.get("app_sp")}
RUN_ID_PARAM = dbutils.widgets.get("run_id")
assert catalog == "mfg_mc_se_sa" and prefix == "bluerridge", "this notebook only touches mfg_mc_se_sa.bluerridge_*"

job_run_id = RUN_ID_PARAM
if not job_run_id or "{{" in job_run_id:
    try: job_run_id = dbutils.notebook.entry_point.getDbutils().notebook().getContext().currentRunId().toString()
    except Exception: job_run_id = "unavailable"
ME = spark.sql("SELECT current_user()").first()[0]
print("job_run_id:", job_run_id, "| raw parameter:", repr(RUN_ID_PARAM))
print("running as:", ME)

SCHEMAS = ["landing", "bronze_project", "bronze_drawings", "silver_engineering", "gold_pm", "agents", "app"]
SCHEMA_FQ = [f"{catalog}.{prefix}_{s}" for s in SCHEMAS]
GOLD, AGENTS, SILVER = f"{catalog}.{prefix}_gold_pm", f"{catalog}.{prefix}_agents", f"{catalog}.{prefix}_silver_engineering"
INTERNAL = r"^(__materialization|event_log)"     # hidden objects Spark Declarative Pipelines creates

def statements(path, params=None):
    """Split a .sql file into statements: drop comment lines, split on semicolons. Returns (sql, skipped_placeholders)."""
    text = "\n".join(l for l in Path(path).read_text().splitlines() if not l.strip().startswith("--"))
    out = []
    for s in [x.strip() for x in text.split(";") if x.strip()]:
        needed = re.findall(r"\{(\w+)\}", s)
        if needed and any(not (params or {}).get(n) for n in needed):
            out.append((s, [n for n in needed if not (params or {}).get(n)]))
        else:
            out.append((s.format(**params) if needed else s, []))
    return out

def run_file(path, params=None):
    done, skipped = [], []
    for s, missing in statements(path, params):
        if missing: skipped.append((s, missing)); continue
        spark.sql(s); done.append(s)
    return done, skipped

def md_table(rows, header):
    return "| " + " | ".join(header) + " |\n|" + "---|" * len(header) + "\n" + "\n".join("| " + " | ".join(str(c) for c in r) + " |" for r in rows)
def is_internal(name): return re.match(INTERNAL, name) is not None

job_run_id: 543672861542223 | raw parameter: '543672861542223'
running as: joseph.hobbs@databricks.com


## 1. Tags

`tags.sql` sets tags on the 7 schemas, 14 managed tables and 3 columns. SDP streaming tables and materialized views are not tagged on purpose (a pipeline refresh could reset them), but their schemas are. `SET TAGS` overwrites, so the file can be re-run.

In [2]:
done, skipped = run_file("tags.sql")
EXPECTED_TAGS = sum(len(re.findall(r"'[^']+'\s*=\s*'[^']+'", s)) for s in done)
print(f"tags.sql: {len(done)} statements applied, {len(skipped)} skipped, {EXPECTED_TAGS} tag assignments")
assert not skipped

tags.sql: 24 statements applied, 0 skipped, 112 tag assignments


## 2. Row filter: build `drawing_register`, the mapping table, the helper and the filter

`row_filters.sql` runs five steps in order: create `drawing_register` (one row per drawing, joining `drawings_extracted` to the per-drawing roll-up of `compliance_findings`), create and seed `discipline_access` by `MERGE`, create `discipline_allowed`, create `rf_drawing_discipline`, then `ALTER TABLE ... SET ROW FILTER`. `latest_status` is FAIL if any rule failed, else REVIEW if the judge could not decide, else PASS. The table is created before the filter exists, so the build itself is not filtered; re-running replaces the table and re-attaches the filter.

In [3]:
done, skipped = run_file("row_filters.sql")
print(f"row_filters.sql: {len(done)} statements applied")
assert len(done) == 7 and not skipped
reg = f"{GOLD}.drawing_register"
desc = spark.sql(f"DESCRIBE TABLE EXTENDED {reg}")
print("DESCRIBE TABLE EXTENDED, the filter and constraint rows:")
display(desc.filter(F.col("col_name").isin("Row Filter", "drawing_register_pk", "Type", "Owner")).select("col_name", "data_type"))
RF_ROW = desc.filter(F.col("col_name") == "Row Filter").first()["data_type"]
assert "rf_drawing_discipline" in RF_ROW and "(discipline)" in RF_ROW, RF_ROW
print("row filter attached:", RF_ROW)

row_filters.sql: 7 statements applied
DESCRIBE TABLE EXTENDED, the filter and constraint rows:


col_name,data_type
Type,MANAGED
Owner,joseph.hobbs@databricks.com
Row Filter,`mfg_mc_se_sa`.`bluerridge_agents`.`rf_drawing_discipline` ON (discipline)
drawing_register_pk,PRIMARY KEY (`drawing_id`)


row filter attached: `mfg_mc_se_sa`.`bluerridge_agents`.`rf_drawing_discipline` ON (discipline)


## 3. Row filter demo, with asserts

Three parts. (a) The helper gives explicit allow and deny answers. (b) What each principal **would** see, as a **simulation** (`discipline_allowed()` applied to the unfiltered source, not a query run as that principal), as row counts per discipline from the unfiltered silver table `drawings_extracted` through `WHERE discipline_allowed('<principal>', discipline)`. (c) What the real filter shows the current user (mapped to `'*'`): all 1,200 rows. The two example.com users cannot log in to this workspace, so (b) is the faithful stand-in for their view.

In [4]:
FN = f"{AGENTS}.discipline_allowed"
CASES = [("mech.engineer@example.com", "piping", False), ("mech.engineer@example.com", "mechanical", True),
         ("piping.lead@example.com", "piping", True), ("piping.lead@example.com", "mechanical", False),
         ("joseph.hobbs@databricks.com", "instr", True), ("joseph.hobbs@databricks.com", "piping", True),
         ("unknown.person@example.com", "piping", False), ("unknown.person@example.com", "mechanical", False)]
rows = []
for p, d, want in CASES:
    got = spark.sql(f"SELECT {FN}('{p}', '{d}')").first()[0]
    rows.append((p, d, want, got))
    assert got is want, f"discipline_allowed({p}, {d}) = {got}, expected {want}"
print("(a) discipline_allowed(principal, discipline): all", len(CASES), "assertions pass")
display(spark.createDataFrame(rows, "principal string, discipline string, expected boolean, actual boolean"))

SRC = f"{SILVER}.drawings_extracted"
src_total = spark.table(SRC).count()
src_by_disc = {r["discipline"]: r["n"] for r in spark.sql(f"SELECT discipline, count(*) n FROM {SRC} GROUP BY discipline").collect()}
print("unfiltered source", SRC, "rows:", src_total, "| by discipline:", dict(sorted(src_by_disc.items(), key=lambda kv: str(kv[0]))))
VISIBLE = {}
for p in ["joseph.hobbs@databricks.com", "mech.engineer@example.com", "piping.lead@example.com", "unknown.person@example.com"]:
    df = spark.sql(f"SELECT coalesce(discipline, '(none)') AS discipline, count(*) AS rows FROM {SRC} WHERE {FN}('{p}', discipline) GROUP BY 1")
    VISIBLE[p] = {r["discipline"]: r["rows"] for r in df.collect()}
print("(b) SIMULATION, rows each principal would see per discipline (helper applied to the unfiltered source, not run as them):")
for p, v in VISIBLE.items(): print(f"  {p:32s} total {sum(v.values()):5d}  {dict(sorted(v.items()))}")
assert sum(VISIBLE["joseph.hobbs@databricks.com"].values()) == 1200 == src_total
assert VISIBLE["mech.engineer@example.com"] == {"mechanical": src_by_disc["mechanical"]}
assert VISIBLE["piping.lead@example.com"] == {"piping": src_by_disc["piping"]}
assert VISIBLE["unknown.person@example.com"] == {}

n_user = spark.sql(f"SELECT COUNT(*) FROM {reg}").first()[0]
by_disc_user = {r["discipline"]: r["n"] for r in spark.sql(f"SELECT coalesce(discipline, '(none)') discipline, count(*) n FROM {reg} GROUP BY 1").collect()}
print(f"(c) as {ME} (mapped to '*'): SELECT COUNT(*) FROM drawing_register =", n_user, "|", dict(sorted(by_disc_user.items())))
assert n_user == 1200
assert sum(by_disc_user.values()) == 1200 and by_disc_user["mechanical"] == src_by_disc["mechanical"]
status = {r["latest_status"]: r["n"] for r in spark.sql(f"SELECT latest_status, count(*) n FROM {reg} GROUP BY 1").collect()}
fail_total = spark.sql(f"SELECT sum(fail_count) FROM {reg}").first()[0]
cf_fail = spark.sql(f"SELECT count(*) FROM {GOLD}.compliance_findings WHERE status = 'FAIL'").first()[0]
print("drawing_register latest_status:", status, "| sum(fail_count):", fail_total, "| FAIL rows in compliance_findings:", cf_fail)
assert fail_total == cf_fail and sum(status.values()) == 1200
assert spark.sql(f"SELECT count(DISTINCT drawing_id) FROM {reg}").first()[0] == 1200
print("mapping table discipline_access:")
display(spark.table(f"{AGENTS}.discipline_access").orderBy("principal"))

(a) discipline_allowed(principal, discipline): all 8 assertions pass


principal,discipline,expected,actual
mech.engineer@example.com,piping,False,False
mech.engineer@example.com,mechanical,True,True
piping.lead@example.com,piping,True,True
piping.lead@example.com,mechanical,False,False
joseph.hobbs@databricks.com,instr,True,True
joseph.hobbs@databricks.com,piping,True,True
unknown.person@example.com,piping,False,False
unknown.person@example.com,mechanical,False,False


unfiltered source mfg_mc_se_sa.bluerridge_silver_engineering.drawings_extracted rows: 1200 | by discipline: {None: 2, 'electrical': 77, 'instr': 119, 'mechanical': 134, 'piping': 755, 'structural': 113}
(b) SIMULATION, rows each principal would see per discipline (helper applied to the unfiltered source, not run as them):
  joseph.hobbs@databricks.com      total  1200  {'(none)': 2, 'electrical': 77, 'instr': 119, 'mechanical': 134, 'piping': 755, 'structural': 113}
  mech.engineer@example.com        total   134  {'mechanical': 134}
  piping.lead@example.com          total   755  {'piping': 755}
  unknown.person@example.com       total     0  {}
(c) as joseph.hobbs@databricks.com (mapped to '*'): SELECT COUNT(*) FROM drawing_register = 1200 | {'(none)': 2, 'electrical': 77, 'instr': 119, 'mechanical': 134, 'piping': 755, 'structural': 113}
drawing_register latest_status: {'REVIEW': 3, 'PASS': 878, 'FAIL': 319} | sum(fail_count): 498 | FAIL rows in compliance_findings: 498
mapping table

principal,discipline
joseph.hobbs@databricks.com,*
mech.engineer@example.com,mechanical
piping.lead@example.com,piping


## 4. Nothing else is filtered or masked

Unity Catalog lists every row filter and column mask in `information_schema.row_filters` and `column_masks`. Across the `bluerridge_*` schemas there must be exactly one filter (on `drawing_register`) and no masks, so the Lakebase sync sources (`kpi_daily`, `compliance_findings`, `risk_forecast`) and every pipeline or notebook input stay unfiltered. `DESCRIBE TABLE EXTENDED` on those three is shown as well.

In [5]:
rf = spark.sql(f"SELECT table_schema, table_name, filter_name, target_columns FROM {catalog}.information_schema.row_filters WHERE table_schema LIKE '{prefix}\\_%'")
cm = spark.sql(f"SELECT table_schema, table_name, column_name, mask_name FROM {catalog}.information_schema.column_masks WHERE table_schema LIKE '{prefix}\\_%'")
print("row filters in bluerridge_* schemas:"); display(rf)
print("column masks in bluerridge_* schemas:", cm.count())
assert [(r["table_schema"], r["table_name"]) for r in rf.collect()] == [(f"{prefix}_gold_pm", "drawing_register")]
assert cm.count() == 0
NO_FILTER = {}
for t in ["kpi_daily", "compliance_findings", "risk_forecast"]:
    d = spark.sql(f"DESCRIBE TABLE EXTENDED {GOLD}.{t}")
    has_rf = d.filter(F.col("col_name") == "Row Filter").count()
    has_mask = d.filter(F.lower(F.col("col_name")).contains("mask")).count()
    NO_FILTER[t] = (has_rf, has_mask)
    print(f"{t}: Row Filter rows in DESCRIBE TABLE EXTENDED = {has_rf}, mask rows = {has_mask}")
    assert has_rf == 0 and has_mask == 0

row filters in bluerridge_* schemas:


table_schema,table_name,filter_name,target_columns
bluerridge_gold_pm,drawing_register,mfg_mc_se_sa.bluerridge_agents.rf_drawing_discipline,discipline


column masks in bluerridge_* schemas: 0
kpi_daily: Row Filter rows in DESCRIBE TABLE EXTENDED = 0, mask rows = 0
compliance_findings: Row Filter rows in DESCRIBE TABLE EXTENDED = 0, mask rows = 0
risk_forecast: Row Filter rows in DESCRIBE TABLE EXTENDED = 0, mask rows = 0


## 5. Grants

`grants.sql` holds the intended grants for the project managers' group, the engineers' group and the app service principal. A statement runs only when its placeholder widget has a value. In this run none was supplied (see "Groups outcome"), so every statement is listed as not applied. Then `SHOW GRANTS` is captured for each schema. The command also returns grants inherited from the catalog, so direct schema grants (ObjectType = SCHEMA) are separated from inherited ones. **Inherited access matters: see section 5b.** The `BROAD` assertion below checks *direct* grants only.

In [6]:
done, skipped = run_file("grants.sql", PARAMS)
print(f"grants.sql: {len(done)} statements applied, {len(skipped)} not applied (placeholder has no value)")
for s, missing in skipped: print("  not applied, needs", missing, ":", s)
GRANTS = {}
for fq in SCHEMA_FQ:
    g = spark.sql(f"SHOW GRANTS ON SCHEMA {fq}").collect()
    direct = [(r["Principal"], r["ActionType"]) for r in g if r["ObjectType"] == "SCHEMA"]
    inherited = len([r for r in g if r["ObjectType"] != "SCHEMA"])
    GRANTS[fq] = (direct, inherited)
    print(f"{fq}: {len(direct)} direct schema grants, {inherited} inherited from the catalog {direct}")
VOL_GRANTS = {}
for v in ["drawings", "project"]:
    g = spark.sql(f"SHOW GRANTS ON VOLUME {catalog}.{prefix}_landing.{v}").collect()
    VOL_GRANTS[v] = [(r["Principal"], r["ActionType"]) for r in g if r["ObjectType"] == "VOLUME"]
    print(f"volume {v}: direct grants {VOL_GRANTS[v]}")
BROAD = ("account users", "users", "admins", "All Account Users")
for fq, (direct, _) in GRANTS.items():
    assert not any(p in BROAD for p, _ in direct), f"direct grant to a broad principal on {fq}"
print("no DIRECT grant on a schema or volume of this project goes to a broad principal (this check does not see inherited catalog grants, see next cell)")

grants.sql: 0 statements applied, 14 not applied (placeholder has no value)
  not applied, needs ['pm_group'] : GRANT USE SCHEMA ON SCHEMA mfg_mc_se_sa.bluerridge_gold_pm TO `{pm_group}`
  not applied, needs ['pm_group'] : GRANT SELECT ON SCHEMA mfg_mc_se_sa.bluerridge_gold_pm TO `{pm_group}`
  not applied, needs ['engineer_group'] : GRANT USE SCHEMA ON SCHEMA mfg_mc_se_sa.bluerridge_silver_engineering TO `{engineer_group}`
  not applied, needs ['engineer_group'] : GRANT SELECT ON SCHEMA mfg_mc_se_sa.bluerridge_silver_engineering TO `{engineer_group}`
  not applied, needs ['engineer_group'] : GRANT USE SCHEMA ON SCHEMA mfg_mc_se_sa.bluerridge_gold_pm TO `{engineer_group}`
  not applied, needs ['engineer_group'] : GRANT SELECT ON SCHEMA mfg_mc_se_sa.bluerridge_gold_pm TO `{engineer_group}`
  not applied, needs ['engineer_group'] : GRANT USE SCHEMA ON SCHEMA mfg_mc_se_sa.bluerridge_landing TO `{engineer_group}`
  not applied, needs ['engineer_group'] : GRANT READ VOLUME ON VOLUME mfg_mc_

## 5b. Inherited access (outside this project's control)

Direct grants are not the whole story. Unity Catalog grants on the catalog `mfg_mc_se_sa` are inherited by every schema, table and Volume below it. This cell reads `SHOW GRANTS ON CATALOG` and the effective inherited rows of `SHOW GRANTS ON SCHEMA` and records what the built-in group `account users` can do. The catalog is shared and owned by others, so this build does not and should not change it. Other catalog-level grantees are only counted, never listed, because the repository is public.

In [7]:
cat_g = spark.sql(f"SHOW GRANTS ON CATALOG {catalog}").collect()
AU = "account users"
au_cat = sorted({r["ActionType"] for r in cat_g if r["Principal"] == AU and r["ObjectType"] == "CATALOG"})
au_meta = sorted({r["ActionType"] for r in cat_g if r["Principal"] == AU and r["ObjectType"] == "METASTORE"})
others_cat = {r["Principal"] for r in cat_g if r["Principal"] != AU and r["ObjectType"] == "CATALOG"}
others_use = {r["Principal"] for r in cat_g if r["Principal"] != AU and r["ObjectType"] == "CATALOG" and r["ActionType"] == "USE CATALOG"}
print("account users, privileges on catalog", catalog + ":", au_cat)
print("account users, privileges inherited from the metastore:", au_meta)
print("other principals with any catalog-level grant:", len(others_cat), "| holding USE CATALOG:", len(others_use))
INHERITED = {}
for fq in SCHEMA_FQ:
    gs = spark.sql(f"SHOW GRANTS ON SCHEMA {fq}").collect()
    INHERITED[fq] = sorted({r["ActionType"] for r in gs if r["Principal"] == AU and r["ObjectType"] != "SCHEMA"})
    print(f"{fq.replace(catalog + '.' + prefix + '_', '')}: privileges reaching account users by inheritance: {INHERITED[fq]}")
assert len(cat_g) > 0, "SHOW GRANTS ON CATALOG returned nothing"
READ_SET = {"USE CATALOG", "USE SCHEMA", "SELECT", "READ VOLUME"}
ACCOUNT_WIDE_READ = READ_SET <= set(au_cat)
print("account users can read every bluerridge_* table and the drawings Volume through the catalog:", ACCOUNT_WIDE_READ)
SCHEMA_READ = {"USE SCHEMA", "SELECT", "READ VOLUME"}   # USE CATALOG is a catalog-level privilege and is not listed on a schema
assert all(SCHEMA_READ <= set(v) for v in INHERITED.values()) == ACCOUNT_WIDE_READ, INHERITED

account users, privileges on catalog mfg_mc_se_sa: ['EXECUTE', 'READ VOLUME', 'SELECT', 'USE CATALOG', 'USE SCHEMA']
account users, privileges inherited from the metastore: ['READ METADATA']
other principals with any catalog-level grant: 42 | holding USE CATALOG: 22
landing: privileges reaching account users by inheritance: ['EXECUTE', 'READ METADATA', 'READ VOLUME', 'SELECT', 'USE SCHEMA']
bronze_project: privileges reaching account users by inheritance: ['EXECUTE', 'READ METADATA', 'READ VOLUME', 'SELECT', 'USE SCHEMA']
bronze_drawings: privileges reaching account users by inheritance: ['EXECUTE', 'READ METADATA', 'READ VOLUME', 'SELECT', 'USE SCHEMA']
silver_engineering: privileges reaching account users by inheritance: ['EXECUTE', 'READ METADATA', 'READ VOLUME', 'SELECT', 'USE SCHEMA']
gold_pm: privileges reaching account users by inheritance: ['EXECUTE', 'READ METADATA', 'READ VOLUME', 'SELECT', 'USE SCHEMA']
agents: privileges reaching account users by inheritance: ['EXECUTE', 'R

## 6. Tag listing

Read back from `information_schema.schema_tags`, `table_tags` and `column_tags`, restricted to the `bluerridge_*` schemas, with SDP's hidden `__materialization_*` and `event_log_*` objects removed. The total must equal the number of assignments in `tags.sql`.

In [8]:
def tags(view, extra=""):
    df = spark.sql(f"SELECT * FROM {catalog}.information_schema.{view} WHERE schema_name LIKE '{prefix}\\_%'")
    return [r.asDict() for r in df.collect() if not ("table_name" in r.asDict() and is_internal(r["table_name"]))]
st, tt, ct = tags("schema_tags"), tags("table_tags"), tags("column_tags")
print(f"schema tags {len(st)} | table tags {len(tt)} | column tags {len(ct)} | total {len(st)+len(tt)+len(ct)} (tags.sql assigns {EXPECTED_TAGS})")
assert len(st) + len(tt) + len(ct) == EXPECTED_TAGS
display(spark.createDataFrame([(r["schema_name"], r["tag_name"], r["tag_value"]) for r in sorted(st, key=lambda r: (r["schema_name"], r["tag_name"]))], "schema_name string, tag_name string, tag_value string"))
display(spark.createDataFrame([(r["schema_name"], r["table_name"], r["tag_name"], r["tag_value"]) for r in sorted(tt, key=lambda r: (r["schema_name"], r["table_name"], r["tag_name"]))], "schema_name string, table_name string, tag_name string, tag_value string"))
display(spark.createDataFrame([(r["schema_name"], r["table_name"], r["column_name"], r["tag_name"], r["tag_value"]) for r in sorted(ct, key=lambda r: (r["table_name"], r["column_name"]))], "schema_name string, table_name string, column_name string, tag_name string, tag_value string"))
assert {r["schema_name"] for r in st} == {f"{prefix}_{s}" for s in SCHEMAS}
assert not [r for r in tt + ct if is_internal(r["table_name"])]

schema tags 39 | table tags 70 | column tags 3 | total 112 (tags.sql assigns 112)


schema_name,tag_name,tag_value
bluerridge_agents,br_data_class,reference
bluerridge_agents,br_domain,document_intelligence
bluerridge_agents,br_layer,agents
bluerridge_agents,br_sensitivity,low
bluerridge_agents,synthetic,true
bluerridge_app,br_data_class,reference
bluerridge_app,br_domain,project_controls
bluerridge_app,br_layer,app
bluerridge_app,br_sensitivity,low
bluerridge_app,synthetic,true


schema_name,table_name,tag_name,tag_value
bluerridge_agents,code_excerpts,br_data_class,reference
bluerridge_agents,code_excerpts,br_domain,document_intelligence
bluerridge_agents,code_excerpts,br_layer,agents
bluerridge_agents,code_excerpts,br_sensitivity,low
bluerridge_agents,code_excerpts,synthetic,true
bluerridge_agents,discipline_access,br_data_class,reference
bluerridge_agents,discipline_access,br_domain,document_intelligence
bluerridge_agents,discipline_access,br_layer,agents
bluerridge_agents,discipline_access,br_sensitivity,high
bluerridge_agents,discipline_access,synthetic,true


schema_name,table_name,column_name,tag_name,tag_value
bluerridge_gold_pm,compliance_findings,evidence_text,br_data_class,ai_output
bluerridge_gold_pm,compliance_findings,reason,br_data_class,ai_output
bluerridge_gold_pm,daily_brief,brief_md,br_data_class,ai_output


## 7. Lineage

`system.access.table_lineage` is queried for edges whose source and target are both `bluerridge_*` tables that **still exist** (lineage keeps dropped scratch tables such as the `_dryrun` ones, which are filtered out by joining to `information_schema.tables`), de-duplicated, with SDP internals removed. Column lineage is added for the two tables the app leans on, `kpi_daily` and `compliance_findings`. System tables lag by minutes to hours, so edges for objects written during this very run (the new `drawing_register`) may be missing; that is noted in the output rather than hidden.

In [9]:
existing = {f"{catalog}.{r['table_schema']}.{r['table_name']}" for r in spark.sql(f"SELECT table_schema, table_name FROM {catalog}.information_schema.tables WHERE table_schema LIKE '{prefix}\\_%'").collect() if not is_internal(r["table_name"])}
edges_df = spark.sql(f"""
  SELECT DISTINCT source_table_full_name AS src, target_table_full_name AS tgt
  FROM system.access.table_lineage
  WHERE source_table_full_name LIKE '{catalog}.{prefix}\\_%' AND target_table_full_name LIKE '{catalog}.{prefix}\\_%'
""")
EDGES = sorted((r["src"], r["tgt"]) for r in edges_df.collect() if r["src"] in existing and r["tgt"] in existing and r["src"] != r["tgt"])
short = lambda n: n.replace(f"{catalog}.{prefix}_", "")
print(f"{len(EDGES)} distinct table-level edges between existing bluerridge tables")
for s, t in EDGES: print(f"  {short(s)} -> {short(t)}")
layer_of = lambda n: short(n).split(".")[0].split("_")[0]
assert any(layer_of(s) == "bronze" and layer_of(t) == "silver" for s, t in EDGES), "no bronze -> silver edge"
assert any(layer_of(s) == "silver" and layer_of(t) == "gold" for s, t in EDGES), "no silver -> gold edge"
assert (f"{GOLD}.compliance_findings", f"{GOLD}.kpi_daily") in EDGES
assert any(t == f"{GOLD}.compliance_findings" for s, t in EDGES) and sum(1 for s, t in EDGES if t == f"{GOLD}.kpi_daily") >= 5
reg_edges = [e for e in EDGES if f"{GOLD}.drawing_register" in e]
print("edges touching drawing_register (new this run, lineage may lag):", reg_edges or "none yet")

col_df = spark.sql(f"""
  SELECT DISTINCT source_table_full_name AS src, source_column_name AS scol, target_table_full_name AS tgt, target_column_name AS tcol
  FROM system.access.column_lineage
  WHERE target_table_full_name IN ('{GOLD}.kpi_daily', '{GOLD}.compliance_findings')
    AND source_table_full_name LIKE '{catalog}.{prefix}\\_%'
""")
COLS = sorted((r["tgt"], r["tcol"], r["src"], r["scol"]) for r in col_df.collect() if r["src"] in existing)
print(f"{len(COLS)} distinct column-level edges into kpi_daily and compliance_findings")

29 distinct table-level edges between existing bluerridge tables
  agents.code_excerpts -> agents.code_excerpts_vs
  bronze_drawings.raw_files -> silver_engineering.drawings_parsed
  bronze_drawings.raw_files -> silver_engineering.drawings_raw_tracking
  bronze_project.change_orders_raw -> silver_engineering.change_orders
  bronze_project.cost_raw -> silver_engineering.cost
  bronze_project.daily_reports_raw -> silver_engineering.daily_reports
  bronze_project.ncrs_raw -> silver_engineering.ncrs
  bronze_project.safety_log_raw -> silver_engineering.safety_log
  bronze_project.schedule_raw -> silver_engineering.schedule
  gold_pm.compliance_findings -> gold_pm.drawing_register
  gold_pm.compliance_findings -> gold_pm.kpi_daily
  gold_pm.compliance_judge_raw -> gold_pm.compliance_judge_raw_v1_baseline
  gold_pm.compliance_judge_raw -> gold_pm.compliance_judge_raw_v2_prompt_hints
  silver_engineering.cost -> gold_pm.kpi_daily
  silver_engineering.daily_reports -> gold_pm.kpi_daily
  silve

## 8. Plan check: no AI calls

The governance work is pure SQL. The formatted physical plans of the register build and the demo query are scanned for `AI Query` steps and must contain none (`explain` plans only and runs nothing).

In [10]:
def ai_steps(sql):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        spark.sql(sql).explain(mode="formatted")
    plan = re.split(r"^== (?!Physical Plan)", buf.getvalue(), flags=re.M)[0]
    return len(re.findall(r"^\(\d+\) AI Query.*$", plan, flags=re.M))
register_select = [s for s, _ in statements("row_filters.sql") if s.startswith("INSERT INTO")][0].split("\n", 1)[1]
PLAN = {"drawing_register build SELECT": ai_steps(register_select),
        "per-principal demo query": ai_steps(f"SELECT discipline, count(*) FROM {SRC} WHERE {FN}('mech.engineer@example.com', discipline) GROUP BY 1"),
        "filtered read of drawing_register": ai_steps(f"SELECT count(*) FROM {reg}")}
for k, v in PLAN.items(): print(f"AI Query steps in the physical plan, {k}: {v}")
assert sum(PLAN.values()) == 0

AI Query steps in the physical plan, drawing_register build SELECT: 0
AI Query steps in the physical plan, per-principal demo query: 0
AI Query steps in the physical plan, filtered read of drawing_register: 0


## 9. Write the evidence files

`outputs/grant_listing.md` (copied to `evidence/grant_listing.md`) and `outputs/lineage_edges.md` are written from the variables above, not typed.

In [11]:
out = Path("outputs"); out.mkdir(exist_ok=True)
L = []
L.append(f"# Unity Catalog governance: grants, tags, row filter\n\nGenerated by `06_governance/apply.ipynb`, job run `{job_run_id}`, as `{ME}`. All objects are under `{catalog}.{prefix}_*`. Synthetic data only.\n")
L.append("## 1. Grants (`SHOW GRANTS ON SCHEMA`)\n")
L.append("Direct grants on each schema are listed; the owner (`joseph.hobbs@databricks.com`) holds everything through ownership and is not shown by `SHOW GRANTS`. \"Inherited\" counts grants that reach the schema from the shared catalog `mfg_mc_se_sa` (not this project's to change).\n")
L.append(md_table([(f"`{short(fq)}`", len(d), "; ".join(f"{p}: {a}" for p, a in d) or "none", inh) for fq, (d, inh) in GRANTS.items()], ["Schema", "Direct grants", "Principal: privilege", "Inherited from catalog"]))
L.append("\nVolumes: " + "; ".join(f"`{catalog}.{prefix}_landing.{v}` direct grants = {len(g)}" for v, g in VOL_GRANTS.items()) + ".\n")
L.append("**No direct grants were made by this task** (asserted for schemas and Volumes: no direct grant to `account users` or any other broad principal). Inherited access is a different matter, see the next section.\n")
L.append("### Inherited access (outside this project's control)\n")
L.append(f"The built-in group `account users` holds these privileges on the catalog `{catalog}`: " + ", ".join(f"`{a}`" for a in au_cat) + (" (plus " + ", ".join(f"`{a}`" for a in au_meta) + " on the metastore)" if au_meta else "") + ". They were granted by the catalog owners or inherited from the metastore, not by this task, and Unity Catalog inherits them to every schema, table and Volume below the catalog.\n")
L.append(md_table([(f"`{short(fq)}`", ", ".join(v) or "none") for fq, v in INHERITED.items()], ["Schema", "Privileges reaching `account users` by inheritance"]) + "\n")
L.append("**Consequence, stated plainly:** " + ("every `bluerridge_*` table (for example `kpi_daily` and `drawings_extracted`) and the drawing Volume is readable by all account principals today. " if ACCOUNT_WIDE_READ else "see the privileges above. ") + "**The row filter on `drawing_register` is the only access control this build adds**, and it protects that one table only. This is acceptable because all data in the project is synthetic. Do not read '0 grants applied' as 'data locked down'.\n")
L.append(f"Other catalog-level grantees, counted only (no names are listed because the repository is public): {len(others_cat)} other principals hold a catalog-level grant, {len(others_use)} of them USE CATALOG (for example colleagues' app service principals).\n")
L.append("### Groups outcome: NOT possible, and why\n")
L.append("- `databricks groups create --display-name bluerridge-pm` and `bluerridge-engineers` succeeded as **workspace-local** groups.")
L.append("- A grant of USE SCHEMA on `mfg_mc_se_sa.bluerridge_gold_pm` to `bluerridge-pm` was rejected: `PRINCIPAL_DOES_NOT_EXIST: Could not find principal with name bluerridge-pm`. Unity Catalog only resolves account-level groups, and creating those needs account admin, which this demo does not have.")
L.append("- Both probe groups were deleted (`databricks groups list` shows no `bluerridge-*` group). This task granted nothing broad instead.")
L.append("- The intended grants are in [`06_governance/grants.sql`](../06_governance/grants.sql) with `{pm_group}`, `{engineer_group}` and `{app_sp}` placeholders. `apply.ipynb` applies a statement when its widget has a value: this run supplied none, so **0 grants were applied**.\n")
L.append("Intended grants (not applied):\n")
L.append(md_table([("project managers group", "USE SCHEMA, SELECT", "`bluerridge_gold_pm`"), ("engineers group", "USE SCHEMA, SELECT", "`bluerridge_silver_engineering`, `bluerridge_gold_pm`"), ("engineers group", "USE SCHEMA, READ VOLUME", "`bluerridge_landing`, Volume `drawings`"), ("project managers group", "USE CATALOG", "catalog owner must grant (shared catalog)")], ["Principal", "Privileges", "On"]))
L.append("\n### TODO for Task 24 (post-deploy)\n")
L.append("- Look up the app's service principal application id and run the `{app_sp}` statements in `grants.sql`: USE SCHEMA + SELECT on `bluerridge_gold_pm` and `bluerridge_agents`, READ VOLUME on `bluerridge_landing.drawings` (it also needs USE CATALOG on `mfg_mc_se_sa` from the catalog owner).")
L.append("- `INSERT INTO mfg_mc_se_sa.bluerridge_agents.discipline_access VALUES ('<app sp application id>', '*')`. Without that row the row filter hides every `drawing_register` row from the app.\n")
L.append("## 2. Row-level security on `drawing_register`\n")
L.append(f"`DESCRIBE TABLE EXTENDED {reg}` shows `Row Filter`: `{RF_ROW}`.\n")
L.append("Design: `discipline_access(principal, discipline)` (`'*'` = all) -> SQL function `discipline_allowed(principal, discipline)` -> row filter function `rf_drawing_discipline(discipline)` = `discipline_allowed(current_user(), discipline)`, attached with `ALTER TABLE ... SET ROW FILTER ... ON (discipline)`.\n")
L.append("### Mapping table\n")
L.append(md_table([(r["principal"], r["discipline"]) for r in spark.table(f"{AGENTS}.discipline_access").orderBy("principal").collect()], ["principal", "discipline"]) + "\n\n`example.com` principals are fictitious.\n")
L.append("### Helper allow/deny results (all asserted)\n")
L.append(md_table([(f"`{p}`", d, w, g) for p, d, w, g in rows], ["principal", "discipline", "expected", "actual"]) + "\n")
L.append("### What each principal would see: a SIMULATION (the helper applied to the unfiltered `drawings_extracted`, not queries run as those principals)\n")
discs = ["electrical", "instr", "mechanical", "piping", "structural", "(none)"]
L.append(md_table([(f"`{p}`", *[v.get(d, 0) for d in discs], sum(v.values())) for p, v in VISIBLE.items()], ["principal", *discs, "total"]) + "\n")
L.append(f"As `{ME}` (mapped to `*`), the real filtered table returns `SELECT COUNT(*) FROM drawing_register` = **{n_user}**. `latest_status`: " + ", ".join(f"{k} {v}" for k, v in sorted(status.items())) + f"; sum(fail_count) = {fail_total} = FAIL rows in `compliance_findings`.\n")
L.append("### Nothing else is filtered or masked\n")
L.append(f"`information_schema.row_filters` for `{prefix}_*`: {rf.count()} row (`drawing_register`). `column_masks`: {cm.count()}. `kpi_daily`, `compliance_findings`, `risk_forecast` (Lakebase sync sources): no `Row Filter` row in `DESCRIBE TABLE EXTENDED`.\n")
L.append("## 3. Tags\n")
L.append("The account enforces governed tags for `layer`, `sensitivity`, `data_class` and `domain` with their own allowed values (for example `sensitivity` only `pii | internal | public`), and rejected the project's values. The project vocabulary therefore lives in `br_layer`, `br_sensitivity`, `br_data_class`, `br_domain` and `synthetic`; the governed `layer` is also set to bronze, silver or gold where it fits. SDP internals (`__materialization_*`, `event_log_*`) are filtered out.\n")
L.append(f"Totals: {len(st)} schema tags, {len(tt)} table tags, {len(ct)} column tags = **{len(st)+len(tt)+len(ct)}** (equals the assignments in `tags.sql`).\n")
L.append("### Schema tags\n")
sch = {}
for r in st: sch.setdefault(r["schema_name"], {})[r["tag_name"]] = r["tag_value"]
keys = ["layer", "br_layer", "br_sensitivity", "br_data_class", "br_domain", "synthetic"]
L.append(md_table([(f"`{s}`", *[v.get(k, "") for k in keys]) for s, v in sorted(sch.items())], ["schema", *[f"`{k}`" for k in keys]]) + "\n")
L.append("### Table tags\n")
tab = {}
for r in tt: tab.setdefault((r["schema_name"], r["table_name"]), {})[r["tag_name"]] = r["tag_value"]
tk = keys[1:]
L.append(md_table([(f"`{s}.{t}`", *[v.get(k, "") for k in tk]) for (s, t), v in sorted(tab.items())], ["table", *[f"`{k}`" for k in tk]]) + "\n")
L.append("### Column tags\n")
L.append(md_table([(f"`{r['schema_name']}.{r['table_name']}.{r['column_name']}`", f"`{r['tag_name']}`", r["tag_value"]) for r in sorted(ct, key=lambda r: (r["table_name"], r["column_name"]))], ["column", "tag", "value"]) + "\n")
L.append("## 4. Lineage\n")
L.append(f"{len(EDGES)} distinct table-level edges between existing `bluerridge_*` tables, in [`06_governance/outputs/lineage_edges.md`](../06_governance/outputs/lineage_edges.md), covering bronze -> silver -> gold including the `compliance_findings` and `kpi_daily` paths.\n")
L.append("## 5. Plan check\n")
L.append("AI Query steps in the physical plans: " + ", ".join(f"{k} = {v}" for k, v in PLAN.items()) + ". This notebook makes no AI calls.\n")
Path("outputs/grant_listing.md").write_text("\n".join(L))

M = [f"# Lineage edges, `{catalog}.{prefix}_*`\n",
     f"Source: `system.access.table_lineage` and `column_lineage`, read by `06_governance/apply.ipynb` (job run `{job_run_id}`). {len(EDGES)} distinct table-to-table edges between tables that exist now; dropped scratch tables (for example `_dryrun`) and SDP hidden objects are excluded. System lineage lags the data by minutes to hours, so `drawing_register` (created in this run) " + ("appears below.\n" if reg_edges else "has no edges yet.\n"),
     "## Diagram\n", "```mermaid", "flowchart LR"]
ids = {}
for s, t in EDGES:
    for n in (s, t): ids.setdefault(n, f"n{len(ids)}")
for n, i in ids.items(): M.append(f'  {i}["{short(n)}"]')
for s, t in EDGES: M.append(f"  {ids[s]} --> {ids[t]}")
M += ["```\n", "## Table-level edges\n", md_table([(f"`{short(s)}`", f"`{short(t)}`") for s, t in EDGES], ["source", "target"]) + "\n"]
for tgt in ("kpi_daily", "compliance_findings"):
    M.append(f"## Paths into `{tgt}`\n")
    M.append(md_table([(f"`{short(s)}`",) for s, t in EDGES if t.endswith("." + tgt)], ["direct sources"]) + "\n")
M.append("## Column-level lineage into `kpi_daily` and `compliance_findings`\n")
M.append(f"{len(COLS)} distinct (source column -> target column) edges.\n")
M.append(md_table([(f"`{short(t)}.{tc}`", f"`{short(s)}.{sc}`") for t, tc, s, sc in COLS], ["target column", "source column"]) + "\n")
Path("outputs/lineage_edges.md").write_text("\n".join(M))
print("wrote outputs/grant_listing.md", len(Path("outputs/grant_listing.md").read_text()), "chars")
print("wrote outputs/lineage_edges.md", len(Path("outputs/lineage_edges.md").read_text()), "chars")

wrote outputs/grant_listing.md 10723 chars
wrote outputs/lineage_edges.md 6828 chars
